# SREDT Pipeline: Real GDELT CSV, GARP 2025 Taxonomy, Venter Dual Criterion

This notebook demonstrates the **SREDT (Sector-Risk Editorial Development Triangle)** pipeline — a method for predicting whether corporate risk scenarios materialize, using a triangle projection model adapted from actuarial science (Venter 1998).

**Key methodological contributions:**
1. **Real GDELT data**: Retrieves actual news articles via GDELT V1 CSV bulk downloads
2. **GARP 2025 L3 taxonomy**: Six risk categories encoded as embedding centroids
3. **GICS sector centroids**: Sub-industry label strings for Energy and Financials
4. **Venter Dual Criterion**: Intercept significance `|a| ≥ 2·SE(a)` is the *primary* criterion
5. **LLM-as-Judge**: Ground truth labels from `meta-llama/llama-3.3-70b-instruct`

**This demo** loads pre-computed results from `mini_demo_data.json` and re-runs the Venter diagnostics, projection, and evaluation steps.


In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# sentence-transformers, psutil, loguru — NOT pre-installed on Colab
_pip('sentence-transformers==3.4.1')
_pip('psutil==6.1.1')
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import gc
import json
import math
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats as scipy_stats
from scipy.stats import spearmanr
from sentence_transformers import SentenceTransformer
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.preprocessing import MinMaxScaler

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-8ea2f1-semantic-risk-evidence-development-trian/main/round-2/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded: {len(data['datasets'][0]['examples'])} examples")
print(f"Data source: {data['metadata']['data_source']}")

## Configuration

Tunable parameters for this demo. Start at minimum values; increase for larger runs.

In [ ]:
# --- Config: minimum values for demo ---
N_DEMO_TRAIN = 6       # number of training examples to use (original: 30)
N_DEMO_TEST  = 5       # number of test examples to use (original: 10)
SIM_THRESHOLD = 0.15   # cosine similarity threshold for cell mass (original: 0.15)
EMBED_BATCH_SIZE = 64  # embedding batch size (original: 64)

## Constants — Fixed Ontologies (GARP 2025 + GICS)

The SREDT pipeline uses two fixed ontologies as embedding anchors:
- **GARP 2025 L3**: Six risk categories with descriptive expansions
- **GICS L1/L2**: Sub-industry label+code strings for Energy and Financials sectors

In [ ]:
GARP_L3_CENTROIDS = [
    ("Operational Risk",  "Operational Risk covering process failures technology outages human errors and legal compliance breaches"),
    ("Business Risk",     "Business Risk covering revenue volatility competitive pressure pricing risk and demand shifts"),
    ("Strategic Risk",    "Strategic Risk covering mergers and acquisitions execution failure business model disruption and capital allocation errors"),
    ("Reputational Risk", "Reputational Risk covering brand damage ESG controversy media scrutiny and stakeholder trust loss"),
    ("Financial Risk",    "Financial Risk covering credit default market price movements liquidity stress and interest rate exposure"),
    ("ESG Risk",          "ESG Risk covering climate transition physical climate hazard social license and governance failures"),
]

GICS_L1_BY_SECTOR = {
    "energy": [
        "Oil Gas Consumable Fuels 101020",
        "Energy Equipment Services 101010",
    ],
    "financials": [
        "Diversified Banks 40101010",
        "Regional Banks 40101015",
        "Asset Management Custody Banks 40203010",
        "Life Health Insurance 40301020",
        "Property Casualty Insurance 40301040",
        "Multi-line Insurance 40301030",
    ],
}

GICS_L2_BY_SECTOR = {
    "energy": ["Energy 1010"],
    "financials": ["Banks 4010", "Financial Services 4020", "Insurance 4030"],
}

GARP_KEYWORDS: dict = {
    "Operational Risk": ["process", "technology", "outage", "compliance", "fraud", "human error"],
    "Business Risk": ["revenue", "competitive", "pricing", "demand"],
    "Strategic Risk": ["merger", "acquisition", "strategy", "disruption", "capital"],
    "Reputational Risk": ["reputation", "brand", "media", "esg", "scandal"],
    "Financial Risk": ["credit", "market", "liquidity", "interest rate", "default"],
    "ESG Risk": ["climate", "carbon", "transition", "governance", "social"],
}

## Load and Inspect Pre-computed Scenario Data

The mini data file contains pre-computed SREDT triangle mass values (L0–L4) for each scenario, plus LLM-judge labels for test scenarios. We extract these to reconstruct the SREDT triangle matrix C.

In [ ]:
examples = data['datasets'][0]['examples']

# Separate train and test scenarios
train_exs = [e for e in examples if e['metadata_split'] == 'train'][:N_DEMO_TRAIN]
test_exs  = [e for e in examples if e['metadata_split'] == 'test'][:N_DEMO_TEST]

print(f"Using {len(train_exs)} train + {len(test_exs)} test scenarios")
print()

# Preview scenarios
df_preview = pd.DataFrame([{
    'split': e['metadata_split'],
    'company': e['metadata_company'],
    'sector': e['metadata_sector'],
    'risk_type': e['metadata_risk_type'],
    'L0': float(e['metadata_l0_mass']),
    'L1': float(e['metadata_l1_mass']),
    'L2': float(e['metadata_l2_mass']),
} for e in train_exs + test_exs])
print(df_preview.to_string(index=False))

## Step 3 — Embedding & Taxonomy Centroids

The SREDT pipeline uses `all-MiniLM-L6-v2` to embed both article texts and taxonomy centroid strings. Here we demonstrate computing GARP L3 centroid embeddings and measuring inter-category distances.

In [ ]:
def embed_strings(model: SentenceTransformer, texts: list) -> np.ndarray:
    if not texts:
        return np.zeros((0, 384), dtype=np.float32)
    embs = model.encode(texts, normalize_embeddings=True, batch_size=EMBED_BATCH_SIZE, show_progress_bar=False)
    return embs.astype(np.float32)


def compute_cell_mass(
    article_embeddings: np.ndarray,
    centroid_embedding: np.ndarray,
    threshold: float = SIM_THRESHOLD,
) -> float:
    if len(article_embeddings) == 0:
        return 0.0
    sims = article_embeddings @ centroid_embedding
    above = sims[sims > threshold]
    if len(above) == 0:
        return 0.0
    return float(np.mean(above) * np.log1p(len(above)))


def compute_sector_centroid(model: SentenceTransformer, centroid_strings: list) -> np.ndarray:
    embs = embed_strings(model, centroid_strings)
    mean_emb = embs.mean(axis=0)
    norm = np.linalg.norm(mean_emb)
    return (mean_emb / (norm + 1e-8)).astype(np.float32)

In [ ]:
print("Loading sentence-transformers model: all-MiniLM-L6-v2")
emb_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded")

# Embed the GARP L3 centroid strings
garp_strings = [s for _, s in GARP_L3_CENTROIDS]
garp_labels  = [name for name, _ in GARP_L3_CENTROIDS]
L3_centroids = embed_strings(emb_model, garp_strings)  # shape: (6, 384)

# Show inter-category cosine similarity matrix
sim_matrix = L3_centroids @ L3_centroids.T
df_sim = pd.DataFrame(sim_matrix, index=garp_labels, columns=garp_labels).round(3)
print("\nGARP L3 inter-category cosine similarity:")
print(df_sim.to_string())

## Step 4 — Reconstruct SREDT Triangle Matrix C

The SREDT triangle C has shape `(n_scenarios, 5)` where columns are L0–L4 mass values:
- **L0**: log(1 + article count)
- **L1**: GICS L1 sub-industry cell mass
- **L2**: GICS L2 industry group cell mass
- **L3**: GARP L3 risk category cell mass (NaN for test)
- **L4**: scenario-text cell mass (NaN for test)

We reconstruct C from pre-computed values in the loaded data.

In [ ]:
def extract_masses(ex: dict, is_test: bool) -> list:
    """Extract L0-L4 mass values from a pre-computed example."""
    l0 = float(ex['metadata_l0_mass'])
    l1 = float(ex['metadata_l1_mass'])
    l2 = float(ex['metadata_l2_mass'])
    if is_test:
        l3 = float('nan')
        l4 = float('nan')
    else:
        l3 = float(ex.get('metadata_l3_mass', 0.0))
        l4 = float(ex.get('metadata_l4_mass', 0.0))
    return [l0, l1, l2, l3, l4]


C_train = np.array([extract_masses(e, is_test=False) for e in train_exs], dtype=float)
C_test  = np.array([extract_masses(e, is_test=True)  for e in test_exs],  dtype=float)
C = np.vstack([C_train, C_test])

print(f"SREDT triangle C shape: {C.shape}")
print(f"C_train stats: mean={np.nanmean(C_train):.4f}, non-zero={np.sum(C_train > 0)}")
print(f"C_test  stats: L0={np.nanmean(C_test[:,0]):.4f}, L1={np.nanmean(C_test[:,1]):.4f}")

df_C = pd.DataFrame(
    C,
    columns=['L0', 'L1', 'L2', 'L3', 'L4'],
    index=[e['metadata_company'] + ' (' + e['metadata_split'] + ')' for e in train_exs + test_exs]
).round(4)
print("\nSREDT triangle matrix C:")
print(df_C.to_string())

## Step 5 — Venter Diagnostics (Dual Criterion)

For each adjacent pair of triangle columns (L0→L1, L1→L2, L2→L3, L3→L4), we run Venter (1998) regression diagnostics:

**Dual criterion** (the key fix over iteration 1):
1. **PRIMARY**: If `|intercept| ≥ 2·SE(intercept)`, use `factor_plus_constant` projection
2. **SECONDARY**: If CV of link ratios < 0.30, use `chain_ladder`; if CV > 0.50, use `bf_fallback`

The primary criterion prevents false chain_ladder verdicts when the intercept is significant.

In [ ]:
def venter_diagnostics(C_train: np.ndarray) -> list:
    """Run Venter (1998) diagnostics for L0→L1, L1→L2, L2→L3, L3→L4 transitions."""
    results = []
    transition_labels = ["L0→L1", "L1→L2", "L2→L3", "L3→L4"]

    for j in range(4):
        x = C_train[:, j]
        y = C_train[:, j + 1]
        n = len(x)

        valid_mask = np.isfinite(x) & np.isfinite(y)
        x = x[valid_mask]
        y = y[valid_mask]
        n = len(x)

        if np.std(x) < 1e-8 or n < 3:
            results.append({
                "transition": transition_labels[j], "f_j": None, "cv": None,
                "intercept": None, "se_intercept": None,
                "intercept_t_stat": None, "intercept_p": None,
                "intercept_significant": False,
                "slope": None, "slope_p": None, "r_squared": None,
                "verdict": "insufficient_data",
                "n_observations": int(n),
            })
            continue

        slope, intercept, r, p_slope, _ = scipy_stats.linregress(x, y)

        x_mean = float(np.mean(x))
        SS_xx = float(np.sum((x - x_mean) ** 2))
        y_hat = intercept + slope * x
        SS_res = float(np.sum((y - y_hat) ** 2))
        s2 = SS_res / max(n - 2, 1)
        se_intercept = float(np.sqrt(s2 * (1.0 / n + x_mean ** 2 / (SS_xx + 1e-12))))

        intercept_t = abs(float(intercept)) / (se_intercept + 1e-12)
        intercept_significant = bool(intercept_t >= 2.0)
        intercept_p = float(2 * (1 - scipy_stats.t.cdf(intercept_t, df=max(n - 2, 1))))

        f_j = float(np.sum(y) / np.sum(x)) if np.sum(x) > 0 else None

        valid_x = x > 0
        if valid_x.sum() >= 2:
            link_ratios = y[valid_x] / x[valid_x]
            mean_lr = float(np.mean(link_ratios))
            cv = float(np.std(link_ratios) / mean_lr) if abs(mean_lr) > 1e-8 else float("inf")
        else:
            cv = float("inf")

        # DUAL CRITERION: intercept significance PRIMARY
        if intercept_significant:
            verdict = "factor_plus_constant"
        elif np.isfinite(cv) and cv < 0.30:
            verdict = "chain_ladder"
        elif np.isfinite(cv) and cv > 0.50:
            verdict = "bf_fallback"
        else:
            verdict = "borderline"

        results.append({
            "transition": transition_labels[j],
            "f_j": f_j,
            "cv": float(cv) if np.isfinite(cv) else None,
            "intercept": float(intercept),
            "se_intercept": se_intercept,
            "intercept_t_stat": float(intercept_t),
            "intercept_p": intercept_p,
            "intercept_significant": intercept_significant,
            "slope": float(slope),
            "slope_p": float(p_slope),
            "r_squared": float(r ** 2),
            "verdict": verdict,
            "n_observations": int(n),
        })

    return results

In [ ]:
diag_results = venter_diagnostics(C_train)

print("Venter Diagnostics Results:")
print(f"{'Transition':<12} {'Intercept':>10} {'SE(a)':>8} {'|t|':>6} {'sig':>5} {'CV':>7} {'Verdict'}")
print("-" * 65)
for d in diag_results:
    a = f"{d['intercept']:.4f}" if d['intercept'] is not None else "N/A"
    se = f"{d['se_intercept']:.4f}" if d['se_intercept'] is not None else "N/A"
    t = f"{d['intercept_t_stat']:.2f}" if d['intercept_t_stat'] is not None else "N/A"
    sig = "YES" if d['intercept_significant'] else "no"
    cv = f"{d['cv']:.3f}" if d['cv'] is not None else "N/A"
    print(f"{d['transition']:<12} {a:>10} {se:>8} {t:>6} {sig:>5} {cv:>7}  {d['verdict']}")

## Step 6 — Projection for Test Scenarios

For test scenarios, L3 and L4 are missing (future data). We project them using the Venter verdict:
- `factor_plus_constant`: `ĉ = intercept + slope × c_obs`
- `chain_ladder`: `ĉ = f_j × c_obs`
- `bf_fallback` (Bornhuetter-Ferguson): `ĉ = c_obs + E_prior × (1 - q̂)` using prior mean from training

In [ ]:
def project_test_scenarios(
    C_test: np.ndarray,
    diag_results: list,
    C_train: np.ndarray,
) -> list:
    """Project missing L3 and L4 for test scenarios."""

    def _project_one(c_obs: float, diag: dict, E_prior: float, q_hat: float) -> tuple:
        v = diag["verdict"]
        if v == "factor_plus_constant" and diag.get("intercept") is not None and diag.get("slope") is not None:
            c_hat = float(diag["intercept"]) + float(diag["slope"]) * c_obs
        elif v == "chain_ladder" and diag.get("f_j") is not None and not np.isnan(diag["f_j"]):
            c_hat = float(diag["f_j"]) * c_obs
        else:
            c_hat = c_obs + E_prior * (1.0 - q_hat)
        return max(0.0, float(c_hat)), v

    E_prior_L3 = float(np.nanmean(C_train[:, 3]))
    E_prior_L4 = float(np.nanmean(C_train[:, 4]))
    q_hat_L3 = float(np.nansum(C_train[:, 2]) / max(np.nansum(C_train[:, 3]), 1e-8))
    q_hat_L4 = float(np.nansum(C_train[:, 3]) / max(np.nansum(C_train[:, 4]), 1e-8))

    diag_L2_L3 = diag_results[2]
    diag_L3_L4 = diag_results[3]

    projections = []
    for i in range(len(C_test)):
        c_l2 = float(C_test[i, 2])
        c_l3, method_l3 = _project_one(c_l2, diag_L2_L3, E_prior_L3, q_hat_L3)
        c_l4, method_l4 = _project_one(c_l3, diag_L3_L4, E_prior_L4, q_hat_L4)
        projections.append({
            "projected_l3": round(c_l3, 6),
            "projected_l4": round(c_l4, 6),
            "projection_method_l2_l3": method_l3,
            "projection_method_l3_l4": method_l4,
        })

    return projections

In [ ]:
projections = project_test_scenarios(C_test, diag_results, C_train)

print("Test scenario projections:")
print(f"{'Company':<18} {'L2_obs':>8} {'L3_proj':>9} {'L4_proj':>9} {'Method_L2→L3'}")
print("-" * 70)
for ex, proj in zip(test_exs, projections):
    print(f"{ex['metadata_company']:<18} {float(ex['metadata_l2_mass']):>8.4f} "
          f"{proj['projected_l3']:>9.4f} {proj['projected_l4']:>9.4f} "
          f"{proj['projection_method_l2_l3']}")

## Step 9 — Evaluation

Evaluate SREDT predictions against LLM-judge ground truth labels. Compare to:
- **Flat cosine baseline**: mean cosine similarity between pre-day45 articles and scenario text
- **Keyword frequency baseline**: fraction of articles matching GARP risk keywords

In [ ]:
def evaluate(
    projected_l4: list,
    labels: list,
    flat_cosines: list,
    keyword_freqs: list,
    C_test_l0: np.ndarray,
    ground_truth_valid: bool,
) -> dict:
    """Compute AUROC, Brier, Spearman for SREDT vs. baselines."""
    l0_vals = list(C_test_l0)
    valid_mask_l0 = [np.isfinite(v) for v in l0_vals]
    l4_clean = [v for v, ok in zip(projected_l4, valid_mask_l0) if ok]
    l0_clean = [v for v, ok in zip(l0_vals, valid_mask_l0) if ok]

    if len(l4_clean) >= 2 and len(l0_clean) >= 2:
        l0_l4_rho = float(spearmanr(l0_clean, l4_clean).statistic)
    else:
        l0_l4_rho = None

    if not ground_truth_valid:
        return {
            "ground_truth_valid": False,
            "n_valid_labels": sum(1 for l in labels if l != -1),
            "sredt_auroc": None, "baseline_cosine_auroc": None, "baseline_kw_auroc": None,
            "sredt_spearman": None, "baseline_cosine_spearman": None,
            "sredt_brier": None, "baseline_cosine_brier": None,
            "l0_l4_rank_corr": l0_l4_rho,
        }

    valid_mask = [l != -1 for l in labels]
    y_true = np.array([l for l, v in zip(labels, valid_mask) if v])
    y_sredt = np.array([s for s, v in zip(projected_l4, valid_mask) if v])
    y_flat = np.array([f for f, v in zip(flat_cosines, valid_mask) if v])
    y_kw = np.array([k for k, v in zip(keyword_freqs, valid_mask) if v])

    def _safe_auroc(y_true_arr, y_score_arr):
        try:
            if len(np.unique(y_true_arr)) < 2:
                return None
            return round(float(roc_auc_score(y_true_arr, y_score_arr)), 4)
        except Exception:
            return None

    def _safe_brier(y_true_arr, y_score_arr):
        try:
            scaler = MinMaxScaler()
            y_norm = np.clip(scaler.fit_transform(y_score_arr.reshape(-1, 1)).ravel(), 0.0, 1.0)
            return round(float(brier_score_loss(y_true_arr, y_norm)), 4)
        except Exception:
            return None

    def _safe_spearman(y_score_arr, y_true_arr):
        try:
            return round(float(spearmanr(y_score_arr, y_true_arr).statistic), 4)
        except Exception:
            return None

    return {
        "ground_truth_valid": True,
        "n_valid_labels": int(len(y_true)),
        "n_materialized": int(np.sum(y_true)),
        "sredt_auroc": _safe_auroc(y_true, y_sredt),
        "baseline_cosine_auroc": _safe_auroc(y_true, y_flat),
        "baseline_kw_auroc": _safe_auroc(y_true, y_kw),
        "sredt_spearman": _safe_spearman(y_sredt, y_true),
        "baseline_cosine_spearman": _safe_spearman(y_flat, y_true),
        "sredt_brier": _safe_brier(y_true, y_sredt),
        "baseline_cosine_brier": _safe_brier(y_true, y_flat),
        "l0_l4_rank_corr": l0_l4_rho,
    }

In [ ]:
# Extract pre-computed baselines and labels from loaded data
projected_l4_list = [p['projected_l4'] for p in projections]
flat_cosines  = [float(e['predict_baseline_cosine']) for e in test_exs]
kw_freqs      = [float(e['predict_baseline_keyword']) for e in test_exs]
labels        = [int(e['metadata_llm_judge_label']) for e in test_exs]

n_valid = sum(1 for l in labels if l != -1)
ground_truth_valid = n_valid >= max(2, len(test_exs) * 0.8)

eval_metrics = evaluate(
    projected_l4=projected_l4_list,
    labels=labels,
    flat_cosines=flat_cosines,
    keyword_freqs=kw_freqs,
    C_test_l0=C_test[:, 0],
    ground_truth_valid=ground_truth_valid,
)
print("Evaluation metrics:")
for k, v in eval_metrics.items():
    print(f"  {k}: {v}")

## Results Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('SREDT Pipeline Results', fontsize=14, fontweight='bold')

# --- 1. Venter diagnostics: CV and intercept t-stat by transition ---
ax1 = axes[0, 0]
valid_diag = [d for d in diag_results if d['cv'] is not None]
transitions = [d['transition'] for d in valid_diag]
cvs = [d['cv'] for d in valid_diag]
tstats = [d['intercept_t_stat'] for d in valid_diag]
colors = ['red' if d['intercept_significant'] else 'steelblue' for d in valid_diag]
x = np.arange(len(transitions))
bars = ax1.bar(x - 0.2, cvs, 0.35, label='CV (link ratios)', color='steelblue', alpha=0.7)
bars2 = ax1.bar(x + 0.2, [t/10 for t in tstats], 0.35, label='|t| / 10 (intercept)', color='orange', alpha=0.7)
ax1.axhline(0.30, color='steelblue', linestyle='--', alpha=0.5, label='CV=0.30 (CL threshold)')
ax1.axhline(0.50, color='steelblue', linestyle=':', alpha=0.5, label='CV=0.50 (BF threshold)')
ax1.axhline(0.20, color='orange', linestyle='--', alpha=0.5, label='|t|/10=0.20 (sig @2.0)')
ax1.set_xticks(x)
ax1.set_xticklabels(transitions)
ax1.set_title('Venter Dual Criterion Diagnostics')
ax1.set_ylabel('Value')
ax1.legend(fontsize=7)
verdicts = [d['verdict'] for d in valid_diag]
for xi, v in zip(x, verdicts):
    ax1.text(xi, max(cvs) * 0.95, v, ha='center', fontsize=7, rotation=15)

# --- 2. SREDT triangle: L0→L4 for training scenarios ---
ax2 = axes[0, 1]
levels = ['L0', 'L1', 'L2', 'L3', 'L4']
for i, ex in enumerate(train_exs):
    row = [C_train[i, j] for j in range(5) if not np.isnan(C_train[i, j])]
    valid_lvls = levels[:len(row)]
    ax2.plot(valid_lvls, row, marker='o', alpha=0.6, linewidth=1.2,
             label=ex['metadata_company'])
ax2.set_title('SREDT Triangle: L0→L4 Mass (Train)')
ax2.set_xlabel('Level')
ax2.set_ylabel('Cell Mass')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.3)

# --- 3. Projected L4 vs. baseline cosine for test scenarios ---
ax3 = axes[1, 0]
companies_test = [e['metadata_company'] for e in test_exs]
test_labels_str = [e['output'] for e in test_exs]
colors_test = ['green' if l == 'MATERIALIZED' else 'gray' for l in test_labels_str]
x_test = np.arange(len(test_exs))
ax3.bar(x_test - 0.2, projected_l4_list, 0.35, label='SREDT projected L4', color='steelblue', alpha=0.7)
ax3.bar(x_test + 0.2, flat_cosines, 0.35, label='Baseline cosine', color='orange', alpha=0.7)
ax3.set_xticks(x_test)
ax3.set_xticklabels([c[:6] for c in companies_test], rotation=45, ha='right', fontsize=8)
ax3.set_title('SREDT vs. Baseline Cosine (Test Scenarios)')
ax3.set_ylabel('Score')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3, axis='y')
# Mark materialized scenarios
for xi, lbl in zip(x_test, test_labels_str):
    if lbl == 'MATERIALIZED':
        ax3.axvspan(xi - 0.45, xi + 0.45, alpha=0.15, color='green')

# --- 4. GARP L3 centroid similarity heatmap ---
ax4 = axes[1, 1]
im = ax4.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax4.set_xticks(range(len(garp_labels)))
ax4.set_yticks(range(len(garp_labels)))
short_labels = [l.replace(' Risk', '') for l in garp_labels]
ax4.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=8)
ax4.set_yticklabels(short_labels, fontsize=8)
plt.colorbar(im, ax=ax4, label='Cosine Similarity')
ax4.set_title('GARP L3 Centroid Similarity Matrix')
for i in range(len(garp_labels)):
    for j in range(len(garp_labels)):
        ax4.text(j, i, f'{sim_matrix[i, j]:.2f}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.savefig('sredt_results.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved to sredt_results.png")

# Summary table
print("\n=== Key Results ===")
meta = data['metadata']
orig_metrics = meta['evaluation_metrics']
print(f"Data source: {meta['data_source']}")
print(f"Train scenarios: {meta['n_train_scenarios']}, Test scenarios: {meta['n_test_scenarios']}")
print(f"\nOriginal (full run) metrics:")
print(f"  SREDT AUROC:           {orig_metrics['sredt_auroc']}")
print(f"  Baseline cosine AUROC: {orig_metrics['baseline_cosine_auroc']}")
print(f"  L0-L4 rank correlation: {orig_metrics['l0_l4_rank_corr']:.4f} (high = circularity)")
print(f"  Materialized: {orig_metrics['n_materialized']}/{orig_metrics['n_valid_labels']} scenarios")
print(f"\nVenter verdicts (full run):")
for t, v in meta['summary']['verdicts_by_transition'].items():
    print(f"  {t}: {v}")